# 面试问题：FlashAttention 怎样用 Online Softmax 在不保存完整注意力矩阵时得到精确结果？

        ## 可直接复述的回答主线

        1. FlashAttention 的关键不是近似注意力，而是改变精确计算的分块顺序以减少高带宽显存读写。
2. 普通实现会物化长度平方的 score 和 probability 矩阵，长上下文时显存代价很高。
3. Online Softmax 为每个 query 维护运行最大值、归一化因子和加权输出，并在新块到来时重标定旧结果。
4. 验证实现应同时比较最终输出误差、分块过程中的最大值和归一化因子，以及中间矩阵峰值。
5. 数值稳定必须减去行最大值，直接对大 logit 做指数会产生无穷和 NaN。
6. 教学 NumPy 实现只解释算法，生产实现还依赖 GPU tiling、融合 kernel、因果 mask 和反向传播。

        后续实验会用同一批输入依次验证朴素方案、核心机制、失败边界和修正效果。

## 1. 真实案例与输入预览

案例是一批客服生成请求，包含请求编号、业务队列、上下文长度、输出上限和 deadline。使用其中一条缩短到 8 Token 的请求构造确定性 Q/K/V，以便完整打印 Online Softmax 状态；长度分布保留真实 Serving 字段，但不代表线上流量。

In [1]:
import math  # 引入注意力缩放因子的平方根计算。
import numpy as np  # 使用基础矩阵运算手写两种注意力实现。
rng = np.random.default_rng(22)  # 固定随机种子以保存确定性教学输出。
requests = [{"id": "cs-101", "queue": "售后", "context_tokens": 128, "max_new_tokens": 48, "deadline_ms": 900}, {"id": "cs-102", "queue": "退款", "context_tokens": 512, "max_new_tokens": 32, "deadline_ms": 1200}, {"id": "cs-103", "queue": "物流", "context_tokens": 2048, "max_new_tokens": 64, "deadline_ms": 1800}, {"id": "cs-104", "queue": "会员", "context_tokens": 4096, "max_new_tokens": 24, "deadline_ms": 2200}, {"id": "cs-105", "queue": "发票", "context_tokens": 8192, "max_new_tokens": 16, "deadline_ms": 3000}]  # 定义五条具有真实 Serving 字段的脱敏请求。
q = rng.normal(size=(8, 4))  # 为可打印的八 Token 请求生成确定性 query。
k = rng.normal(size=(8, 4))  # 为同一请求生成确定性 key。
v = rng.normal(size=(8, 3))  # 为同一请求生成三维 value 表示。
print("教学实验输入请求")  # 标记下表是离线案例而非线上流量。
for request in requests:  # 逐条展示请求长度和 deadline。
    print(f"{request['id']} queue={request['queue']:<2} context={request['context_tokens']:>4} new={request['max_new_tokens']:>2} deadline={request['deadline_ms']}ms")  # 输出当前请求的可读字段。
print(f"缩小演示张量：Q{q.shape} K{k.shape} V{v.shape}，分块大小=2")  # 展示核心实现实际处理的张量形状。

教学实验输入请求
cs-101 queue=售后 context= 128 new=48 deadline=900ms
cs-102 queue=退款 context= 512 new=32 deadline=1200ms
cs-103 queue=物流 context=2048 new=64 deadline=1800ms
cs-104 queue=会员 context=4096 new=24 deadline=2200ms
cs-105 queue=发票 context=8192 new=16 deadline=3000ms
缩小演示张量：Q(8, 4) K(8, 4) V(8, 3)，分块大小=2


## 2. Baseline / 基线：物化完整 score 与 probability 矩阵

普通稳定 softmax 很直接，也作为精确性金标准；问题是长度为 L 时要保存 L×L 中间量。下面同时打印第一行 score、概率和矩阵字节数。

In [2]:
def stable_softmax(matrix):  # 对二维 score 矩阵实现逐行稳定 softmax。
    row_max = np.max(matrix, axis=1, keepdims=True)  # 读取每行最大值防止指数溢出。
    shifted = matrix - row_max  # 把每行最大 logit 平移到零。
    numerator = np.exp(shifted)  # 计算稳定的未归一化概率。
    denominator = np.sum(numerator, axis=1, keepdims=True)  # 汇总每行归一化因子。
    return numerator / denominator  # 返回每行和为一的概率矩阵。
scale = 1.0 / math.sqrt(q.shape[1])  # 按 head dimension 计算注意力缩放。
full_scores = q @ k.T * scale  # 物化完整八乘八注意力 score。
full_probabilities = stable_softmax(full_scores)  # 对完整 score 做稳定 softmax。
baseline_output = full_probabilities @ v  # 得到普通注意力输出作为精确基线。
print("Baseline 第一条 query 的中间量")  # 标记当前输出属于完整矩阵实现。
print("score =", np.round(full_scores[0], 4).tolist())  # 展示第一行缩放后 score。
print("prob  =", np.round(full_probabilities[0], 4).tolist())  # 展示第一行归一化概率。
print(f"完整 score+prob 中间量={full_scores.nbytes + full_probabilities.nbytes} bytes")  # 输出长度平方内存开销。

Baseline 第一条 query 的中间量
score = [-0.5946, 0.9778, -0.4437, -3.395, -2.6422, 0.3086, 1.3732, 0.9236]
prob  = [0.0468, 0.2256, 0.0544, 0.0028, 0.006, 0.1155, 0.335, 0.2137]
完整 score+prob 中间量=1024 bytes


## 3. 底层实现：分块 Online Softmax

每到一个 K/V 块，就更新行最大值 m、归一化因子 l，并按新最大值重新缩放旧输出。trace 会保存第一条 query 的块级状态，直接展示算法如何累积。

In [3]:
def online_attention(query, key, value, block_size):  # 用分块和在线归一化实现精确注意力。
    rows = query.shape[0]  # 读取 query 数量用于初始化逐行状态。
    running_max = np.full(rows, -np.inf)  # 为每条 query 初始化负无穷运行最大值。
    running_sum = np.zeros(rows)  # 为每条 query 初始化指数和。
    running_output = np.zeros((rows, value.shape[1]))  # 初始化归一化后的加权输出。
    trace = []  # 保存第一条 query 的块级状态供教学观察。
    for start in range(0, key.shape[0], block_size):  # 按固定块宽遍历全部 K/V。
        key_block = key[start:start + block_size]  # 读取当前 key 块。
        value_block = value[start:start + block_size]  # 读取与 key 对齐的 value 块。
        block_scores = query @ key_block.T * scale  # 只物化 query 乘当前块的 score。
        block_max = np.max(block_scores, axis=1)  # 计算当前块内逐行最大值。
        new_max = np.maximum(running_max, block_max)  # 合并旧块和新块的全局行最大值。
        old_scale = np.exp(running_max - new_max)  # 计算旧累计量迁移到新最大值坐标的缩放。
        block_weights = np.exp(block_scores - new_max[:, None])  # 在新最大值坐标计算当前块权重。
        new_sum = old_scale * running_sum + np.sum(block_weights, axis=1)  # 合并旧指数和与当前块指数和。
        weighted_old = old_scale[:, None] * running_sum[:, None] * running_output  # 把旧归一化输出还原成加权和并重标定。
        weighted_new = block_weights @ value_block  # 计算当前块对加权和的贡献。
        running_output = (weighted_old + weighted_new) / new_sum[:, None]  # 得到处理到当前块后的归一化输出。
        running_max = new_max  # 提交新的逐行最大值供下一块使用。
        running_sum = new_sum  # 提交新的逐行归一化因子供下一块使用。
        trace.append({"block": f"{start}:{start + len(key_block)}", "m": running_max[0], "l": running_sum[0], "out0": running_output[0, 0]})  # 记录第一条 query 的关键状态。
    return running_output, trace  # 返回最终精确输出和块级轨迹。
flash_output, online_trace = online_attention(q, k, v, block_size=2)  # 对演示张量执行四个 K/V 块。
print("Online Softmax 第一条 query 的块级轨迹")  # 标记下表用于解释 m 和 l 的演化。
print("块       running_m  running_l  output[0]")  # 输出轨迹字段标题。
for item in online_trace:  # 逐块展示在线归一化状态。
    print(f"{item['block']:<7} {item['m']:>9.4f} {item['l']:>10.4f} {item['out0']:>10.4f}")  # 输出当前块处理后的状态。

Online Softmax 第一条 query 的块级轨迹
块       running_m  running_l  output[0]
0:2        0.9778     1.2075    -0.6281
2:4        0.9778     1.4615    -0.4529
4:6        0.9778     2.0004    -0.1337
6:8        1.3732     2.9850    -0.2625


## 4. 结果表与结果解读

最终输出应与普通稳定注意力在浮点误差内一致；区别在于中间 score 峰值从 L×L 降为 L×block。这里打印误差和不同上下文长度的理论中间量。

In [4]:
max_error = float(np.max(np.abs(flash_output - baseline_output)))  # 计算在线实现与完整矩阵实现的最大绝对误差。
memory_rows = []  # 收集五种上下文长度的中间矩阵估算。
for request in requests:  # 对真实长度分布比较平方矩阵和分块矩阵。
    length = request["context_tokens"]  # 读取当前请求上下文长度。
    full_bytes = 2 * length * length * 2  # 按 FP16 score 和 probability 两张矩阵估算字节数。
    tiled_bytes = length * min(128, length) * 2  # 按 128 Token K/V 块估算单张 score 峰值。
    memory_rows.append((request["id"], length, full_bytes / 1024 ** 2, tiled_bytes / 1024 ** 2))  # 保存可读 MiB 结果。
print(f"最终输出最大绝对误差={max_error:.3e}")  # 展示两种实现的精确性对照。
print("请求      长度   完整中间量MiB  分块score峰值MiB")  # 输出长度和内存结果表头。
for request_id, length, full_mib, tiled_mib in memory_rows:  # 逐请求展示理论中间量。
    print(f"{request_id:<8} {length:>5} {full_mib:>14.2f} {tiled_mib:>17.2f}")  # 输出同一长度下的内存对照。
print("解读：Online Softmax 没有改变答案，只改变中间量的驻留方式；长度越长，平方矩阵差距越明显。")  # 解释误差与内存结果的含义。

最终输出最大绝对误差=2.220e-16
请求      长度   完整中间量MiB  分块score峰值MiB
cs-101     128           0.06              0.03
cs-102     512           1.00              0.12
cs-103    2048          16.00              0.50
cs-104    4096          64.00              1.00
cs-105    8192         256.00              2.00
解读：Online Softmax 没有改变答案，只改变中间量的驻留方式；长度越长，平方矩阵差距越明显。


## 5. 失败案例与修正

直接计算 exp(score) 在大 logit 下会溢出。故意给 score 加上 1000，朴素实现产生非有限值；减去行最大值后概率仍正常。

In [5]:
extreme_scores = full_scores[:2, :4] + 1000.0  # 构造大 logit 以复现指数溢出。
with np.errstate(over="ignore", invalid="ignore"):  # 暂停预期数值警告以保持保存输出简洁。
    naive_exp = np.exp(extreme_scores)  # 执行没有最大值平移的错误指数计算。
    naive_probabilities = naive_exp / np.sum(naive_exp, axis=1, keepdims=True)  # 计算会出现无穷除无穷的错误概率。
fixed_probabilities = stable_softmax(extreme_scores)  # 用行最大值平移修正同一输入。
print("错误行为：朴素大 logit softmax 全部有限 =", bool(np.isfinite(naive_probabilities).all()))  # 展示溢出导致的非有限结果。
print("修正行为：稳定 softmax 全部有限 =", bool(np.isfinite(fixed_probabilities).all()), "行和=", np.round(fixed_probabilities.sum(axis=1), 6).tolist())  # 展示修正后的合法概率。

错误行为：朴素大 logit softmax 全部有限 = False
修正行为：稳定 softmax 全部有限 = True 行和= [1.0, 1.0]


## 6. 生产边界

NumPy 代码没有实现 GPU shared memory tiling、因果或 padding mask、dropout、反向传播和混合精度误差分析。生产替换点是融合 kernel，并用真实长度分布检查吞吐和峰值显存。

In [6]:
production_gaps = ["因果与padding mask", "反向传播", "GPU融合kernel", "混合精度误差", "真实长度吞吐"]  # 列出从教学算法到生产内核仍缺少的能力。
print("生产替换清单：", "、".join(production_gaps))  # 明确当前结果不能代表生产 FlashAttention 吞吐。

生产替换清单： 因果与padding mask、反向传播、GPU融合kernel、混合精度误差、真实长度吞吐


## 7. 最小回归测试

断言只验证样本数量、精确性、概率归一化和分块内存趋势。

In [7]:
assert len(requests) >= 5  # 保证案例包含至少五条真实字段请求。
assert max_error < 1.0e-10  # 保证 Online Softmax 与完整稳定注意力数值一致。
assert np.allclose(fixed_probabilities.sum(axis=1), 1.0)  # 保证稳定修正后的每行概率归一化。
assert all(tiled_mib < full_mib for _, length, full_mib, tiled_mib in memory_rows if length > 128)  # 保证长上下文下分块 score 峰值低于完整中间量。
assert len(online_trace) == 4  # 保证八个 key 按块宽二产生四步可观察轨迹。